In [ ]:
# @title Install from fork with uv
REPO_URL = "https://github.com/W-Nana/stream-translator-gpt.git" # @param {type:"string"}
BRANCH = "main" # @param {type:"string"}
extra_features = "webui,hf_asr,qwen_asr,nemo_asr,firered_vad" # @param {type:"string"}
# @markdown Comma-separated extras. Defaults to all extras: `webui,hf_asr,qwen_asr,nemo_asr,firered_vad`.

import os
import shutil
import subprocess
from pathlib import Path

repo_dir = Path("/content/stream-translator-gpt")
if repo_dir.exists():
    shutil.rmtree(repo_dir)

subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(repo_dir)], check=True)
os.chdir(repo_dir)

subprocess.run(
    'mkdir -p .local/bin && curl -LsSf https://astral.sh/uv/install.sh | env UV_INSTALL_DIR="$PWD/.local/bin" sh',
    shell=True,
    check=True,
)

uv = repo_dir / ".local/bin/uv"
extras = []
for name in [item.strip() for item in extra_features.split(",") if item.strip()]:
    extras.extend(["--extra", name])

subprocess.run([str(uv), "python", "install", "3.12"], check=True)
subprocess.run([str(uv), "sync", "--python", "3.12", *extras], check=True)

In [ ]:
# @title Run command line translator
URL = "https://www.youtube.com/watch?v=r1dytEjbUqo" # @param {type:"string"}
asr_backend = "faster_whisper" # @param ["whisper", "faster_whisper", "simul_streaming", "openai_transcription_api", "qwen3_asr", "nemo_asr"]
model = "large" # @param ["small", "medium", "large", "turbo"] {allow-input: true}
language = "ja" # @param {type:"string"}
qwen3_asr_model = "Qwen/Qwen3-ASR-0.6B" # @param {type:"string"}
nemo_asr_model = "nvidia/parakeet-tdt_ctc-0.6b-ja" # @param {type:"string"}
vad_backend = "silero" # @param ["silero", "firered"]
translation_engine = "none" # @param ["none", "openai", "google"]
openai_api_key = "" # @param {type:"string"}
google_api_key = "" # @param {type:"string"}
translation_prompt = "Translate from Japanese to Chinese, only output the translation, do not output the original text" # @param {type:"string"}
translation_history_size = 2 # @param {type:"integer"}
show_latency_log = True # @param {type:"boolean"}
enable_subtitle_sharing = False # @param {type:"boolean"}
subtitle_share_public_port = 8765 # @param {type:"integer"}

import os
import shlex
import subprocess
from pathlib import Path

repo_dir = Path("/content/stream-translator-gpt")
os.chdir(repo_dir)
uv = repo_dir / ".local/bin/uv"

cmd = [
    "stream-translator-gpt",
    URL,
    "--language",
    language,
    "--translation_history_size",
    str(translation_history_size),
]

if asr_backend in {"whisper", "faster_whisper", "simul_streaming"}:
    cmd.extend(["--model", model])
if asr_backend == "faster_whisper":
    cmd.append("--use_faster_whisper")
elif asr_backend == "simul_streaming":
    cmd.append("--use_simul_streaming")
elif asr_backend == "openai_transcription_api":
    cmd.append("--use_openai_transcription_api")
    if openai_api_key:
        cmd.extend(["--openai_api_key", openai_api_key])
elif asr_backend == "qwen3_asr":
    cmd.extend(["--use_qwen3_asr", "--qwen3_asr_model", qwen3_asr_model])
elif asr_backend == "nemo_asr":
    cmd.extend(["--use_nemo_asr", "--nemo_asr_model", nemo_asr_model])

if vad_backend == "firered":
    cmd.extend(["--vad_backend", "firered"])

if translation_engine != "none":
    cmd.extend(["--translation_prompt", translation_prompt])
    if translation_engine == "openai" and openai_api_key:
        cmd.extend(["--openai_api_key", openai_api_key])
    elif translation_engine == "google" and google_api_key:
        cmd.extend(["--google_api_key", google_api_key])

if show_latency_log:
    cmd.append("--show_latency_log")

if enable_subtitle_sharing:
    cmd.extend([
        "--enable_subtitle_sharing",
        "--subtitle_share_host",
        "0.0.0.0",
        "--subtitle_share_public_port",
        str(subtitle_share_public_port),
    ])

print(shlex.join([str(uv), "run", "--no-sync", *cmd]))
subprocess.run([str(uv), "run", "--no-sync", *cmd], check=True)